In [23]:
import os
from dotenv import load_dotenv
from datetime import datetime
import json

from anthropic import Anthropic
from anthropic.types import ToolParam, Message

In [24]:
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(api_key=anthropic_api_key)

model = "claude-sonnet-4-5-20250929"

In [25]:
def add_user_message(messages, message):
    user_message = {
        "role": "user", 
        "content": message.content if isinstance(message, Message) else message}
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant", 
        "content": message.content if isinstance(message, Message) else message}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [26]:
# Tools and schemas
def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [27]:
def get_current_datetime(date_format="%Y-%m-%d %H-%M-%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current local date and time as a formatted string. Use this whenever the user asks for the current date, time, day of the week, or anything that depends on 'now' (e.g., 'what's today's date?', 'what time is it?'). The time comes from the server's local clock and does not include time zone information unless the format string requests it (e.g., %Z). Formatting follows Python strftime directives. If no format is provided, the result looks like '2026-09-24 14-30-05' (note the hyphens in the time portion). The format string must not be empty.",
    "input_schema": {
        "type": "object",
        "properties": {
        "date_format": {
            "type": "string",
            "description": "A Python strftime format string that controls the output. Examples: '%Y-%m-%d' for '2026-09-24', '%H:%M' for '14:30', '%A, %B %d, %Y' for 'Thursday, September 24, 2026'. Defaults to '%Y-%m-%d %H-%M-%S' if omitted. Must be a non-empty string.",
            "default": "%Y-%m-%d %H-%M-%S",
            "minLength": 1
        }
        },
        "required": []
    }
})

In [28]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?"
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_018FqDgQP7ukHi4zvz26uhBQ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [29]:
response.content[0].id

'toolu_018FqDgQP7ukHi4zvz26uhBQ'

In [30]:
result = get_current_datetime(**response.content[0].input)

In [31]:
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": response.content[0].id,
            "content": result,
            "is_error": False
        }
    ]
})

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_018FqDgQP7ukHi4zvz26uhBQ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_018FqDgQP7ukHi4zvz26uhBQ',
    'content': '09:58:25',
    'is_error': False}]}]

In [32]:
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

Message(id='msg_011CfdmMSFycimGv6SUc6siQ', container=None, content=[TextBlock(citations=None, text='The exact time is **09:58:25**.', type='text')], model='claude-sonnet-4-5-20250929', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=910, output_tokens=14, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)

In [33]:
messages = []

add_user_message(messages, "What's the curret time in HH:MM:SS format?")
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

add_assistant_message(messages, response)

In [34]:
messages

[{'role': 'user', 'content': "What's the curret time in HH:MM:SS format?"},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01BaRBz5qftuQoo6GwTHg8ES', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [49]:
def run_batch(invocations = []):
    batch_output = []

    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])

        tool_output = run_tool(name, args)

        batch_output.append({"tool_name": name, "output": tool_output})
        
    return batch_output

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }

        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [50]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
            batch_tool_schema
        ])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [46]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment.  Its 177 days after Jan 1st, 2027."
)

run_conversation(messages)

I'll help you set a reminder for your doctor's appointment. Let me first calculate the date that is 177 days after January 1st, 2027.
Let me calculate this differently. 177 days from January 1st, 2027:
- January: 31 days (remaining: 177 - 31 = 146)
- February: 28 days (2027 is not a leap year, remaining: 146 - 28 = 118)
- March: 31 days (remaining: 118 - 31 = 87)
- April: 30 days (remaining: 87 - 30 = 57)
- May: 31 days (remaining: 57 - 31 = 26)
- June: 26 days

So 177 days after January 1st, 2027 is June 27th, 2027.
----
Setting the following reminder for 2027-06-27T09:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **June 27th, 2027 at 9:00 AM** (177 days after January 1st, 2027). 

Note: I set the reminder for 9:00 AM - if you'd like a different time or need any changes to the reminder, just let me know!


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment.  Its 177 days after Jan 1st, 2027.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. Let me first calculate the date that is 177 days after January 1st, 2027.", type='text'),
   ToolUseBlock(id='toolu_01NaJivFHdcNKzywJF2Z5PUS', caller=DirectCaller(type='direct'), input={'datetime_str': '2027-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01NaJivFHdcNKzywJF2Z5PUS',
    'content': "Error: name 'timedelta' is not defined",
    'is_error': True}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Let me calculate this differently. 177 days from January 1st, 2027:\n- January: 31 days (remaining: 177 - 31 = 146)\n- February: 28 days (2027 is not a leap year, rema

In [51]:
messages = []

add_user_message(
    messages,
    """Set two reminderss for Jan 1, 2027 at 9AM Pacific:
        * I have a doctors appointment
        * Taxes are due
    """,
)

run_conversation(messages)

I'll set up both reminders for January 1, 2027 at 9:00 AM Pacific Time. I'll convert this to ISO 8601 format using PST (UTC-8).
----
Setting the following reminder for 2027-01-01T09:00:00:
I have a doctors appointment
----
----
Setting the following reminder for 2027-01-01T09:00:00:
Taxes are due
----
Perfect! I've set up both reminders for January 1, 2027 at 9:00 AM Pacific Time:
1. "I have a doctors appointment"
2. "Taxes are due"

Both reminders will notify you at the same time on that date.


[{'role': 'user',
  'content': 'Set two reminderss for Jan 1, 2027 at 9AM Pacific:\n        * I have a doctors appointment\n        * Taxes are due\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll set up both reminders for January 1, 2027 at 9:00 AM Pacific Time. I'll convert this to ISO 8601 format using PST (UTC-8).", type='text'),
   ToolUseBlock(id='toolu_01E9mgxZA6uC9FZpMrgt1F1o', caller=DirectCaller(type='direct'), input={'invocations': [{'name': 'set_reminder', 'arguments': '{"content": "I have a doctors appointment", "timestamp": "2027-01-01T09:00:00"}'}, {'name': 'set_reminder', 'arguments': '{"content": "Taxes are due", "timestamp": "2027-01-01T09:00:00"}'}]}, name='batch_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01E9mgxZA6uC9FZpMrgt1F1o',
    'content': '[{"tool_name": "set_reminder", "output": null}, {"tool_name": "set_reminder", "output": null}]',
    'is_error': False}]},
